In [1]:
#imports
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from datasets import load_dataset, concatenate_datasets

/Users/rubenhayrapetyan/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
#Global Variables
PEFT = False
LORA = False
QUANTIZE = False
QUANTIZE_BITS = 4
MODEL = "Qwen/Qwen3-8B"

In [3]:
tasks = [
    "narrativeqa", "qasper", "multifieldqa_en", "multifieldqa_zh", 
    "hotpotqa", "2wikimqa", "musique", "dureader", "gov_report", 
    "qmsum", "multi_news", "vcsum", "trec", "triviaqa", "samsum",
    "lsht", "passage_count", "passage_retrieval_en", "passage_retrieval_zh",
    "lcc", "repobench-p"
]
datasets = {}
for task in tasks:
    datasets[task] = load_dataset("THUDM/LongBench", task, split="test")

In [8]:
#dataset generation
def format(training_example):
    prompt = (
        "Given the following context, answer the question concisely.\n\n"
        f"Question: {training_example['input']}\n"
        f"Context: {training_example['context']}\n\n"
    )
    answer = training_example["answers"]

    if isinstance(answer, list):
        answer = answer[0]
    elif answer == [""] or "":
        raise ValueError("Empty answer found in the dataset.")

    return {"prompt": prompt, "response": answer}

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(training_example):
    prompt_tokens = tokenizer(training_example["prompt"])['input_ids']
    answer_tokens = tokenizer(training_example["response"])['input_ids']
    input_ids = prompt_tokens + answer_tokens + [tokenizer.eos_token_id]
    labels = [-100] * len(prompt_tokens) + answer_tokens + [tokenizer.eos_token_id]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}

In [9]:
train_datasets = {}
for task_name, ds in datasets.items():
    train_datasets[task_name] = ds.map(format, remove_columns=ds.column_names)
all_train_dataset = concatenate_datasets(list(train_datasets.values()))
tokenized_dataset = all_train_dataset.map(tokenize, remove_columns=all_train_dataset.column_names)

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/150 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/4750 [00:00<?, ? examples/s]

In [24]:
train, test = tokenized_dataset.train_test_split(test_size=0.1).values()

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    trust_remote_code=True,
    device_map="auto",
    device_map="auto"
)

In [ ]:
from datasets import load_metric

metric = load_metric("rouge")

def compute_metrics(predictions, references):
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_refs = tokenizer.batch_decode(references, skip_special_tokens=True)
    result = metric.compute(predictions=decoded_preds, references=decoded_refs, use_stemmer=True)
    result = {key: value.mid.fmeasure * 100 for key, value in result.items()}
    return result

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./Qwen3-8B-LongBench-finetuned",
    do_eval=True,
    eval_strategy="steps",
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    gradient_checkpointing=True,
    fp16=False, #only make true with an MPS gpu
    save_steps=600,
    eval_steps=300,
    logging_steps=300,
    learning_rate=5e-5,
    warmup_steps=500,
    save_total_limit=2,
    push_to_hub=False
)

from transformers import Trainer

trainer = Trainer(
    model=model,
    data_collator=lambda data: {
        'input_ids': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['input_ids']) for f in data], batch_first=True, padding_value=tokenizer.pad_token_id
        ),
        'attention_mask': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['attention_mask']) for f in data], batch_first=True, padding_value=0
        ),
        'labels': torch.nn.utils.rnn.pad_sequence(
            [torch.tensor(f['labels']) for f in data], batch_first=True, padding_value=-100
        ),
    },
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train,
    eval_dataset=test,
    tokenizer=tokenizer
)

